# Day 28 — DENSE Error Analysis & Reproduction Repair

**Goal:** focus on checking real failures, making minimal code repairs, and getting the reproduction path stable.

Workflow:

```text
failure / traceback
→ locate exact code path
→ classify cause
→ minimal patch
→ syntax check
→ small smoke run
→ scale experiment
→ preserve diff/result
```

Known starting point:

- Day 26 smoke baseline (`average`, GCN, 5 bundles): **0.4022**
- Day 27 repaired `ranking` path and tested refinement.
- Small multi-stage/refinement experiment reached **0.4742** with 5 bundles.
- Scaling to 100 bundles exposed `APITimeoutError` / `429 rate_limit_rpm_exceeded`.
- The released repo has already shown several interface/reproduction inconsistencies.

Today we do **not** redesign DENSE and do **not** ask lots of reflection questions.

## 1. Freeze current state

In [ ]:
!git branch --show-current
!git rev-parse HEAD
!git status --short
!git diff --stat
!mkdir -p reproduction_logs
!git diff -- bundle.py queryhelper.py utils.py > reproduction_logs/day28_start.diff

## 2. Inspect the current blocker — query reliability

First inspect request, cache, retry, and concurrency code. Do not touch the GNN yet.

In [ ]:
!grep -n -A 90 "def generate" queryhelper.py
!grep -n "ThreadPoolExecutor\|max_workers\|batch_bundle_query" bundle.py
!grep -Rn --exclude-dir=.git "cache_file\|disable_cache\|pickle.dump\|pickle.load" queryhelper.py bundle.py

Target behavior:

```text
cache hit → return cached response

cache miss
→ API request
→ timeout / 429
→ wait/backoff/retry
→ valid string response
→ cache response
```

Keep `max_workers=1` while the current provider is RPM-limited. Increasing concurrency would make the observed 429 problem worse.

## 3. Check API environment without exposing secrets

In [ ]:
import os
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL", "<not set>"))

## 4. Check cache-path robustness

If `args.cache_file` contains a directory, ensure that directory exists **before** opening the pickle file.

Minimal compatibility pattern, only if current code lacks it:

```python
cache_dir = os.path.dirname(args.cache_file)
if cache_dir:
    os.makedirs(cache_dir, exist_ok=True)
```

This fixes an environment/file-system failure; it does not alter DENSE's algorithm.

## 5. Validate the provider response contract

In [ ]:
!grep -n -A 80 "def query" queryhelper.py

`query()` applies regex parsing, so `generate()` must return a non-empty string.

A minimal guard is:

```python
response = completion.choices[0].message.content
if not isinstance(response, str) or not response.strip():
    raise ValueError("Empty or non-string response")
```

If retry/backoff already exists, keep one retry implementation rather than stacking another one.

## 6. Verify the repaired ranking loss

In [ ]:
!grep -n -A 45 "def bundle_loss" bundle.py

For `ranking`, verify that the current code contains both ideas:

```text
L_R  = ranking penalty:
       LLM bundle class should be top-ranked

L_BE = bundle-level CE:
       averaged bundle logits should match LLM bundle label

loss = L_R + L_BE
```

The released invalid CE form lacked a target. The minimal repaired CE term should conceptually be:

```python
F.cross_entropy(torch.mean(bundle_logits, dim=1), bundle_classes)
```

Do not redesign the loss today.

## 7. Verify multi-stage refinement

In [ ]:
!grep -n -A 45 "def solve" bundle.py
!grep -n -A 80 "def bundle_resample" bundle.py || true
!grep -n -A 80 "def bundle_refine" bundle.py || true

Expected control flow:

```text
bundle_presample
→ optimize stage 1
→ refine
→ optimize stage 2
→ refine
→ optimize stage 3
→ evaluate
```

For `--stages 300 100 100`, refinement belongs between stages.

## 8. Mandatory static checks after edits

In [ ]:
!python -m py_compile bundle.py queryhelper.py utils.py
!git diff --check

## 9. Regression run — 5 bundles first

Run this in the terminal for live output:

In [ ]:
cmd = '''python bundle.py \
  --device 0 \
  --dataset cora \
  --bundle_size 5 \
  --num_samples 5 \
  --sample_criterion neighbor \
  --max_hop 2 \
  --query_type gpt \
  --model deepseek/deepseek-v4.1-flash \
  --loss_type ranking \
  --gnn_type gcn \
  --stages 10 \
  --lr 0.001 \
  --wd 0.001 \
  --repeat 1'''
print(cmd)

Pass condition:

```text
valid bundles > 0
no response/parser crash
ranking loss executes
evaluation completes
```

This run checks regression only; its accuracy is not an official reproduction result.

## 10. Intermediate load test — 20 bundles

In [ ]:
cmd = '''python bundle.py \
  --device 0 \
  --dataset cora \
  --bundle_size 5 \
  --num_samples 20 \
  --sample_criterion neighbor \
  --max_hop 2 \
  --query_type gpt \
  --model deepseek/deepseek-v4.1-flash \
  --loss_type ranking \
  --gnn_type gcn \
  --stages 10 \
  --lr 0.001 \
  --wd 0.001 \
  --repeat 1'''
print(cmd)

If 20 bundles still produce 429/timeouts:

```text
DO NOT modify DENSE model/loss
→ keep one worker
→ inspect/increase retry + exponential backoff
→ make sure successful queries are cached
→ distinguish provider failure from invalid LLM label
```

Only scale to 100 after this layer is stable.

## 11. Target reproduction-oriented run

In [ ]:
cmd = '''python bundle.py \
  --device 0 \
  --dataset cora \
  --bundle_size 5 \
  --num_samples 100 \
  --sample_criterion neighbor \
  --max_hop 2 \
  --query_type gpt \
  --model deepseek/deepseek-v4.1-flash \
  --loss_type ranking \
  --gnn_type gcn \
  --stages 300 100 100 \
  --lr 0.001 \
  --wd 0.001 \
  --repeat 1'''
print(cmd)

This is still **not** automatically the paper's official Cora reproduction because the query LLM/backbone and released README/code discrepancies remain. Its purpose is to make our patched pipeline stable at the intended workload.

## 12. First-error protocol

If a run fails, capture it before editing:

```bash
python bundle.py ... 2>&1 | tee reproduction_logs/day28_target.log

grep -n -m 1 -E "Traceback|Error|Exception|429|timed out"     reproduction_logs/day28_target.log
```

Use this classification:

```text
APITimeout / 429              → provider/infrastructure
missing cache/file directory  → compatibility/environment
TypeError / signature error   → interface/repository
NotImplementedError           → unsupported released path
loss/math execution error     → implementation discrepancy
completed but weak accuracy   → scientific error analysis
```

Day28 Conclusion

The repaired DENSE pipeline successfully completed
a 100-bundle Cora experiment using GPT-4o-2024-08-06.

Bundle valid rate: 100.0%
Bundle class accuracy: 76.0%
Final node classification accuracy: 73.99%

The pipeline completed two refinement stages:
5 → 4 → 3.

This establishes the first stable large-scale
reproduction checkpoint.

Remaining question:
Which components explain the final performance,
and how does this configuration differ from the
paper-reported setup?

## 14. Preserve today's repair

In [ ]:
!git status --short
!git diff --check
!git diff -- bundle.py queryhelper.py utils.py

After the fix is actually tested:

```bash
git add bundle.py queryhelper.py
git commit -m "Improve DENSE reproduction query robustness"
git push origin fix/reproduction
```

Never commit API keys, datasets, cache files, or accidental generated artifacts.

# Day 28 Completion Checklist

- [x] Preserve Day 27 starting diff
- [x] Inspect query/cache/retry path
- [x] Fix only confirmed query compatibility problems
- [x] Verify ranking loss repair
- [x] Verify refinement path
- [x] Pass syntax checks
- [x] Pass 5-bundle regression
- [x] Test 20-bundle API load
- [x] Attempt 100 bundles only if query layer is stable
- [x] Record first real blocker or final result
- [x] Commit tested repair

**Exit criterion:** we can clearly distinguish an API/provider failure, a repository implementation failure, and a genuine model-performance result — and the patched pipeline is closer to a stable reproducible run.